In [ ]:
# ── Cell 0: 🔬 Spectral Research Environment Setup ─────────────────
import subprocess
subprocess.run(['pip', 'install', '-q', 'bitsandbytes', 'wandb', 'datasets', 'transformers'], check=True)
subprocess.run(['git', 'clone', 'https://github.com/ey3lock3r/EFV-nn', '/kaggle/working/EFV-nn'], capture_output=True)
import sys
sys.path.insert(0, '/kaggle/working/EFV-nn/src')
print('✅ Research environment ready.')


In [ ]:
# ── Cell 1: 📦 Imports ─────────────────────────────────────────────
import os, gc, time, math
import torch
import torch.nn.functional as F
import bitsandbytes as bnb
import wandb
from transformers import AutoTokenizer
from datasets import load_dataset
from efv_nn.ppc_sharded import ShardedPPCGraphLLM
from efv_nn.research.spectral_sharded import SpectralShardedPPCGraphLLM
from efv_nn.ppc_gnn import spectral_guardian_penalty
print('✅ Imports OK.')


In [ ]:
# ── Cell 2: 📊 Data Pipeline ────────────────────────────────────────
VOCAB_SIZE = 128256

tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3-8B',
                                          token=os.environ.get('HF_TOKEN'))

def get_dataloader(tokenizer, batch_size=2, seq_len=256):
    ds = load_dataset('HuggingFaceFW/fineweb-edu', name='sample-350BT',
                      split='train', streaming=True)
    def _gen():
        buf = []
        for row in ds:
            toks = tokenizer(row['text'])['input_ids']
            buf.extend(toks)
            while len(buf) >= (seq_len + 1) * batch_size:
                chunk = buf[:(seq_len + 1) * batch_size]
                buf   = buf[(seq_len + 1) * batch_size:]
                yield torch.tensor(chunk).reshape(batch_size, seq_len + 1)
    return _gen()

print('✅ Data pipeline ready.')


In [ ]:
# ── Cell 3: ⚙️ Research Configuration ──────────────────────────────
HIDDEN, LAYERS, EXPERTS = 1024, 24, 64

# ── Research Toggles ────────────────────────────────────────────────
ENABLE_SPECTRAL_GUARDIAN  = True   # Pillar 2: Laplacian regularization
ENABLE_SPECTRAL_GATE      = True   # Pillar 1: FFT-based expert routing
ENABLE_EIGEN_RESONANCE    = False  # Pillar 3: EXPERIMENTAL. Start False.

SPECTRAL_LAMBDA = 0.01    # Guardian penalty strength
LOCAL_ITERS     = 24      # PPC reasoning depth
LR              = 8e-5    # Stabilized learning rate

print(f'Pillars active — Gate: {ENABLE_SPECTRAL_GATE} | '
      f'Guardian: {ENABLE_SPECTRAL_GUARDIAN} | '
      f'Eigen: {ENABLE_EIGEN_RESONANCE}')


In [ ]:
# ── Cell 4: 🏗️ Spectral Model Instantiation ─────────────────────────
gc.collect(); torch.cuda.empty_cache()

model = SpectralShardedPPCGraphLLM(
    VOCAB_SIZE, HIDDEN, LAYERS, EXPERTS,
    use_jacobian        = True,
    use_spectral_gate   = ENABLE_SPECTRAL_GATE,
    use_eigen_resonance = ENABLE_EIGEN_RESONANCE,
)
print(f'✅ Spectral Model Ready. Params: {model.total_params/1e9:.2f}B')


In [ ]:
# ── Cell 5: 🔬 Spectral Training Loop ──────────────────────────────
def train_spectral(model, tokenizer, num_steps=1000):
    gc.collect(); torch.cuda.empty_cache()
    wandb.init(project='ppc-spectral-research', name=f'spectral-{time.strftime("%H%M")}')

    dataloader = get_dataloader(tokenizer)
    opt = bnb.optim.PagedAdamW8bit(model.parameters(), lr=LR)

    t0 = time.time(); last_step = 0
    print(f'🔬 Spectral Training [Iters: {LOCAL_ITERS}, LR: {LR}]...')

    for step, batch in enumerate(dataloader):
        ids, targets = batch[:, :-1], batch[:, 1:].to(model.device1)

        opt.zero_grad()
        with torch.amp.autocast('cuda'):
            logits, avg_iters, avg_energy, layer_energies = model(ids, local_iters=LOCAL_ITERS)
            loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))

        # Spectral Guardian: OUTSIDE autocast to preserve FP32 precision
        if ENABLE_SPECTRAL_GUARDIAN:
            loss = loss + spectral_guardian_penalty(layer_energies.float(), lam=SPECTRAL_LAMBDA)

        (loss / 256.0).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0 / 256.0)
        opt.step()

        if step % 5 == 0:
            dt = (time.time() - t0) * 1000 / (step - last_step + 1e-9)
            t0 = time.time(); last_step = step
            wandb.log({'loss': loss.item(), 'avg_energy': avg_energy.item(),
                       'layer_energy_std': layer_energies.std().item(),
                       'duration_ms': dt}, step=step)
            print(f'St {step} | L: {loss.item():.4f} | E: {avg_energy.item():.4f} '
                  f'| E_std: {layer_energies.std().item():.5f} | D: {dt:.0f}ms')

        if step >= num_steps: break

    print(f'\n[Spectral Snapshot] Step {step} | L: {loss.item():.4f} | E_std: {layer_energies.std().item():.5f}')


In [ ]:
# ── Cell 6: 🚀 Launch Spectral Research Run ─────────────────────────
train_spectral(model, tokenizer, num_steps=2000)
